In [1]:
# 03 - pdf parsing (v2): read the fake statements from notebook 02 and check the extractor
# v2: no text cleaning, the description comes out exactly as printed (description_sa),
# then features.add_features() adds the same columns the model was trained on
import re
import pdfplumber
from pathlib import Path
import sys
import pandas as pd

# the code lives in src/ (repo) or next to this notebook (local copy)
sys.path += ["../src", "src", "."]

# find test.csv from notebook 01, same idea as notebook 01 finding the raw csv
PLACES = [Path("../data/processed"), Path("data/processed"), Path("."),
          Path.home() / "finsight_processed", Path.home() / "Downloads" / "finsight_processed"]
PROCESSED = next((p for p in PLACES if (p / "test.csv").exists()), None)
if PROCESSED is None:
    raise FileNotFoundError("test.csv not found - run notebook 01 first, or put test.csv next to this notebook")

# statements go in data/finsight_statements in the repo, otherwise next to test.csv
STMT_DIR = PROCESSED.parent / "finsight_statements" if PROCESSED.name == "processed" else PROCESSED / "finsight_statements"
print("test data :", PROCESSED.resolve())
print("statements:", STMT_DIR.resolve())

from pdf_extraction import extract_transactions, ExtractionError
from features import add_features

index = pd.read_csv(STMT_DIR / "statements_index.csv")
print(f"{len(index)} statements found")

test data : /tmp/claude-0/-home-claude/6f9bbf43-f057-5995-bc15-4e1aeee517ad/scratchpad/v2/data/processed
statements: /tmp/claude-0/-home-claude/6f9bbf43-f057-5995-bc15-4e1aeee517ad/scratchpad/v2/data/finsight_statements
30 statements found


In [2]:
# try one statement from each bank first
for f in index.groupby("bank")["pdf_file"].first():
    ex = extract_transactions(STMT_DIR / "pdfs" / f)
    print(f"{f}: {len(ex)} transactions | opening balance R{ex.attrs['opening_balance']:,.2f} | "
          f"balance check passed on {ex['balance_ok'].mean():.0%} of rows")
    display(ex.head(4))

STMT003_highveld.pdf: 39 transactions | opening balance R9,388.17 | balance check passed on 100% of rows


,page,date,description_sa,amount,balance,balance_ok
0,1,2023-08-01,External Withdrawal Assurance Americ EFT Payme...,-505.75,8882.42,True
1,1,2023-08-02,3168 CHK PURCH SIG ABSA CASHSEND*OLIVIA* . / 3168,-370.00,8512.42,True
2,1,2023-08-02,PURCHASE 0801 WONGA REPAYMENT 1036 WC 31683168,-1864.54,6647.88,True
3,1,2023-08-02,Qt,-34.00,6613.88,True


STMT002_karoo.pdf: 39 transactions | opening balance R12,623.47 | balance check passed on 100% of rows


,page,date,description_sa,amount,balance,balance_ok
0,1,2023-07-01,"Out of Network ATM fee 2518 E. LOGAN STREET, O...",-11.10,12612.37,True
1,1,2023-07-01,TOTALENERGIES/EXPRESS 2978 MAGNOLIA WC,-65.75,12546.62,True
2,1,2023-07-01,Viewfreescor*,-110.82,12435.80,True
3,1,2023-07-01,Withdrawal / 1 KENNEWICK WA BURGER KING F11485...,-38.89,12396.91,True


STMT001_protea.pdf: 43 transactions | opening balance R8,514.39 | balance check passed on 100% of rows


,page,date,description_sa,amount,balance,balance_ok
0,1,2023-06-01,PMNT SENT 0531 STD BANK INSTANT MONEY*ANONYMOU...,-129.50,8384.89,True
1,1,2023-06-02,STD BANK INSTANT MONEY,-370.00,8014.89,True
2,1,2023-06-04,ABSA CASHSEND*BROOKLYN MALL,-277.50,7737.39,True
3,1,2023-06-05,HK JACKSON ENERGY FLAGSTAFF NC 06/04,-51.98,7685.41,True


In [3]:
# why we didnt just use pdfplumber's extract_table()
# it needs table lines to find the columns, so check how many rows it gets per layout
baseline = []
for bank, g in index.groupby("bank"):
    r = g.iloc[0]
    found = 0
    with pdfplumber.open(STMT_DIR / "pdfs" / r.pdf_file) as pdf:
        for page in pdf.pages:
            for table in page.extract_tables():
                found += sum(1 for row in table[1:] if row and row[0] and re.match(r"\d", row[0]))
    expected = r.n_transactions + 1   # +1 for the opening balance row
    baseline.append({"bank": r.bank_name, "layout": r.layout, "extract_table rows": found,
                     "expected rows": expected, "found %": round(100 * found / expected, 1)})
pd.DataFrame(baseline)

,bank,layout,extract_table rows,expected rows,found %
0,Highveld Mutual,zebra,40,40,100.0
1,Karoo Savings Bank,open,0,40,0.0
2,Protea Bank,ruled,44,44,100.0


In [4]:
# run the extractor on all 30 statements and compare with the ground truth
results, extracted_all = [], []
for r in index.itertuples():
    ex = extract_transactions(STMT_DIR / "pdfs" / r.pdf_file)
    gt = pd.read_csv(STMT_DIR / "ground_truth" / r.ground_truth_file)
    n = min(len(ex), len(gt))
    e, g = ex.iloc[:n].reset_index(drop=True), gt.iloc[:n]
    results.append({
        "statement_id": r.statement_id, "bank": r.bank_name, "layout": r.layout,
        "rows_expected": len(gt), "rows_extracted": len(ex),
        "date_acc": (e["date"].astype(str) == g["date"].astype(str)).mean(),
        "amount_acc": ((e["amount"] - g["amount"]).abs() < 0.005).mean(),
        "balance_acc": ((e["balance"] - g["balance"]).abs() < 0.005).mean(),
        # most important one: the text must match description_sa exactly (what the model trained on)
        "description_acc": (e["description_sa"] == g["description_sa"]).mean(),
        "balance_check": e["balance_ok"].mean(),
        "opening_ok": abs(ex.attrs["opening_balance"] - r.opening_balance) < 0.005,
    })
    # keep the true category next to each row so the classifier can be scored on the pdf rows too
    extracted_all.append(ex.assign(statement_id=r.statement_id, bank=r.bank,
                                   category=g["category"].values if len(ex) == len(gt) else None,
                                   is_spend=g["is_spend"].values if len(ex) == len(gt) else None))

res = pd.DataFrame(results)
metrics = ["date_acc", "amount_acc", "balance_acc", "description_acc", "balance_check"]
print(f"Rows: expected {res.rows_expected.sum():,} | extracted {res.rows_extracted.sum():,} | "
      f"opening balances correct: {res.opening_ok.sum()}/{len(res)}")
summary = res.groupby(["bank", "layout"])[metrics].mean().mul(100).round(2).reset_index()
overall = pd.DataFrame([{"bank": "ALL", "layout": "", **res[metrics].mean().mul(100).round(2).to_dict()}])
pd.concat([summary, overall], ignore_index=True)

Rows: expected 1,210 | extracted 1,210 | opening balances correct: 30/30


,bank,layout,date_acc,amount_acc,balance_acc,description_acc,balance_check
0,Highveld Mutual,zebra,100.0,100.0,100.0,100.0,100.0
1,Karoo Savings Bank,open,100.0,100.0,100.0,100.0,100.0
2,Protea Bank,ruled,100.0,100.0,100.0,100.0,100.0
3,ALL,,100.0,100.0,100.0,100.0,100.0


In [5]:
# add the same feature columns as the training data (dates, time, online, txn_type, amount_zar)
extracted = add_features(pd.concat(extracted_all, ignore_index=True))
spend = extracted[extracted["is_spend"] == True]
print(f"{len(extracted):,} rows extracted, {len(spend):,} are spending rows for the classifier")
print("has purchase date:", round(spend["has_purchase_date"].mean() * 100, 1), "%")
print("online:", round(spend["is_online"].mean() * 100, 1), "%")
spend[["statement_id", "date", "description_sa", "amount_zar", "txn_type", "has_purchase_date", "category"]].sample(8, random_state=3)

1,210 rows extracted, 1,162 are spending rows for the classifier
has purchase date: 29.7 %
online: 3.7 %


,statement_id,date,description_sa,amount_zar,txn_type,has_purchase_date,category
368,STMT009,2023-06-30,Pos Debit- 2875 2875 BP 1036 Reston US,80.40,card,0,Transport
864,STMT022,2023-07-20,EXT WD FINCHOICE Inc - 3168,29.60,other,0,Banking
1023,STMT026,2023-07-02,SASOL DELIGHT,126.36,other,0,Transport
150,STMT004,2023-09-27,Withdrawal to,222.00,atm,0,Shopping
427,STMT012,2023-09-05,BURGER KING 5016 BRONX LP 09/02,32.97,other,1,Food
1110,STMT028,2023-09-10,The Box,47.58,other,0,Food
163,STMT005,2023-06-06,PICK N PAY HYPER 3168 RIVERSIDE WC 06/06,34.30,other,1,Groceries
613,STMT016,2023-09-15,POS SIG 09/14 VISA #4711 E3168ON BOCA DEL MAR'...,24.83,card,1,Groceries


In [6]:
# bad files should give a proper error message instead of crashing the app
from reportlab.pdfgen import canvas

tests_dir = STMT_DIR / "error_tests"
tests_dir.mkdir(exist_ok=True)
c = canvas.Canvas(str(tests_dir / "not_a_statement.pdf")); c.drawString(100, 700, "Dear customer, this is a letter."); c.save()
c = canvas.Canvas(str(tests_dir / "blank_like_a_scan.pdf")); c.showPage(); c.save()

for f in sorted(tests_dir.glob("*.pdf")):
    try:
        extract_transactions(f)
        print(f"{f.name}: extracted (unexpected)")
    except ExtractionError as err:
        print(f"{f.name}: rejected -> {err}")

blank_like_a_scan.pdf: rejected -> No text found. Scanned (image) statements are not supported.
not_a_statement.pdf: rejected -> No transaction table found (expected a header with Date, Description and Balance).


In [7]:
# save everything
# extracted_all.csv = what the classifier gets from the pdfs (spending rows: is_spend == True)
extracted.to_csv(STMT_DIR / "extracted_all.csv", index=False)
res.to_csv(STMT_DIR / "extraction_evaluation.csv", index=False)
print(f"saved extracted_all.csv ({len(extracted):,} rows) and extraction_evaluation.csv to {STMT_DIR}")
print(list(extracted.columns))

saved extracted_all.csv (1,210 rows) and extraction_evaluation.csv to ../data/finsight_statements
['page', 'date', 'description_sa', 'amount', 'balance', 'balance_ok', 'statement_id', 'bank', 'category', 'is_spend', 'stmt_year', 'stmt_month', 'stmt_day', 'purchase_year', 'purchase_month', 'purchase_day', 'days_to_process', 'has_purchase_date', 'hour', 'minute', 'has_time', 'is_online', 'txn_type', 'amount_zar']
